In [1]:
# ============================================================================
# Oral Cancer Dataset Download and Setup in Google Drive
# Destination: Google Drive -> /Langraph Projects/Oral Cancer/
# ============================================================================

import os
import zipfile
import requests
from pathlib import Path
from tqdm import tqdm
import shutil
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from IPython.display import display
import numpy as np

In [2]:
# ============================================================================
# STEP 1: MOUNT GOOGLE DRIVE
# ============================================================================

def mount_google_drive():
    """Mount Google Drive"""
    print("=" * 70)
    print("STEP 1: MOUNTING GOOGLE DRIVE")
    print("=" * 70)

    from google.colab import drive
    drive.mount('/content/drive')

    print("\n✅ Google Drive mounted successfully!")
    print(f"📁 Drive location: /content/drive/MyDrive/")
    print("-" * 70)

# ============================================================================
# STEP 2: CREATE DIRECTORY STRUCTURE
# ============================================================================

def create_directories():
    """Create necessary directories in Google Drive"""
    print("\n" + "=" * 70)
    print("STEP 2: CREATING DIRECTORY STRUCTURE")
    print("=" * 70)

    # Define the target directory
    base_path = Path("/content/drive/MyDrive/Langraph Projects/Oral Cancer")

    # Create directory if it doesn't exist
    base_path.mkdir(parents=True, exist_ok=True)

    print(f"\n✅ Created directory: {base_path}")
    print("-" * 70)

    return base_path

# ============================================================================
# STEP 3: DOWNLOAD DATASET
# ============================================================================

def download_dataset(url, destination):
    """Download dataset from S3 with progress bar"""
    print("\n" + "=" * 70)
    print("STEP 3: DOWNLOADING DATASET FROM AWS S3")
    print("=" * 70)

    print(f"\n📥 Source: {url}")
    print(f"📁 Destination: {destination}")
    print("\n⏳ Downloading... This may take several minutes (file is ~3GB)\n")

    try:
        # Send GET request
        response = requests.get(url, stream=True)
        response.raise_for_status()

        # Get total file size
        total_size = int(response.headers.get('content-length', 0))

        # Download with progress bar
        with open(destination, 'wb') as file, tqdm(
            desc="Downloading",
            total=total_size,
            unit='iB',
            unit_scale=True,
            unit_divisor=1024,
            bar_format='{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}]'
        ) as pbar:
            for data in response.iter_content(chunk_size=1024):
                size = file.write(data)
                pbar.update(size)

        print(f"\n✅ Download complete!")
        print(f"📊 File size: {os.path.getsize(destination) / (1024**3):.2f} GB")
        print("-" * 70)

        return True

    except Exception as e:
        print(f"\n❌ Download failed: {str(e)}")
        return False

# ============================================================================
# STEP 4: EXTRACT DATASET
# ============================================================================

def extract_dataset(zip_path, extract_to):
    """Extract the ZIP file"""
    print("\n" + "=" * 70)
    print("STEP 4: EXTRACTING DATASET")
    print("=" * 70)

    print(f"\n📦 Extracting: {zip_path.name}")
    print(f"📁 Extract to: {extract_to}")
    print("\n⏳ Extracting files...\n")

    try:
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            # Get list of files
            file_list = zip_ref.namelist()
            total_files = len(file_list)

            print(f"Total files in archive: {total_files}")

            # Extract with progress
            for idx, file in enumerate(file_list, 1):
                zip_ref.extract(file, extract_to)

                # Show progress every 50 files or at the end
                if idx % 50 == 0 or idx == total_files:
                    print(f"  Progress: {idx}/{total_files} files extracted ({(idx/total_files)*100:.1f}%)", end='\r')

            print(f"\n\n✅ Extraction complete! {total_files} files extracted")

        print("-" * 70)
        return True

    except Exception as e:
        print(f"\n❌ Extraction failed: {str(e)}")
        return False

# ============================================================================
# STEP 5: DELETE ZIP FILE
# ============================================================================

def delete_zip_file(zip_path):
    """Delete the ZIP file to save space"""
    print("\n" + "=" * 70)
    print("STEP 5: CLEANUP - DELETING ZIP FILE")
    print("=" * 70)

    try:
        file_size_gb = os.path.getsize(zip_path) / (1024**3)
        print(f"\n🗑️  Deleting: {zip_path.name}")
        print(f"   Size: {file_size_gb:.2f} GB")

        os.remove(zip_path)

        print(f"✅ ZIP file deleted - {file_size_gb:.2f} GB space freed!")
        print("-" * 70)

    except Exception as e:
        print(f"⚠️  Could not delete ZIP file: {str(e)}")

# ============================================================================
# STEP 6: ANALYZE FILE STRUCTURE
# ============================================================================

def analyze_directory_structure(base_path, max_depth=3):
    """Analyze and display the directory structure"""
    print("\n" + "=" * 70)
    print("STEP 6: ANALYZING FILE STRUCTURE")
    print("=" * 70)

    def get_directory_tree(path, prefix="", depth=0, max_depth=3):
        """Recursively build directory tree"""
        if depth > max_depth:
            return []

        tree_lines = []

        try:
            # Get all items in directory
            items = sorted(path.iterdir(), key=lambda x: (not x.is_dir(), x.name))

            for idx, item in enumerate(items):
                is_last = idx == len(items) - 1
                current_prefix = "└── " if is_last else "├── "

                if item.is_dir():
                    # Count files in directory
                    try:
                        file_count = len(list(item.rglob('*.*')))
                        tree_lines.append(f"{prefix}{current_prefix}📁 {item.name}/ ({file_count} files)")

                        # Recurse into subdirectory
                        extension = "    " if is_last else "│   "
                        tree_lines.extend(get_directory_tree(item, prefix + extension, depth + 1, max_depth))
                    except:
                        tree_lines.append(f"{prefix}{current_prefix}📁 {item.name}/")
                else:
                    # Show file with size
                    size_kb = item.stat().st_size / 1024
                    if size_kb < 1024:
                        size_str = f"{size_kb:.1f} KB"
                    else:
                        size_str = f"{size_kb/1024:.1f} MB"

                    tree_lines.append(f"{prefix}{current_prefix}📄 {item.name} ({size_str})")

        except Exception as e:
            tree_lines.append(f"{prefix}❌ Error reading directory: {str(e)}")

        return tree_lines

    print(f"\n📂 Directory Structure: {base_path}\n")
    print(f"📁 {base_path.name}/")

    tree = get_directory_tree(base_path, max_depth=2)  # Limit depth for readability
    for line in tree[:50]:  # Show first 50 lines
        print(line)

    if len(tree) > 50:
        print(f"\n... and {len(tree) - 50} more items")

    print("\n" + "-" * 70)

# ============================================================================
# STEP 7: DETAILED STATISTICS
# ============================================================================

def show_dataset_statistics(base_path):
    """Show detailed statistics about the dataset"""
    print("\n" + "=" * 70)
    print("STEP 7: DATASET STATISTICS")
    print("=" * 70)

    # Count files by extension
    file_types = {}
    total_files = 0
    total_size = 0
    directories = []

    print("\n📊 Analyzing dataset contents...\n")

    # Walk through directory
    for item in base_path.rglob('*'):
        if item.is_file():
            total_files += 1
            total_size += item.stat().st_size

            # Count by extension
            ext = item.suffix.lower()
            if ext:
                file_types[ext] = file_types.get(ext, 0) + 1
        elif item.is_dir() and item != base_path:
            directories.append(item)

    # Display statistics
    print("=" * 70)
    print("OVERALL STATISTICS")
    print("=" * 70)
    print(f"Total Files:       {total_files}")
    print(f"Total Directories: {len(directories)}")
    print(f"Total Size:        {total_size / (1024**3):.2f} GB")

    print("\n" + "=" * 70)
    print("FILE TYPES")
    print("=" * 70)
    for ext, count in sorted(file_types.items(), key=lambda x: x[1], reverse=True):
        print(f"{ext:15} : {count:5} files")

    print("\n" + "-" * 70)

# ============================================================================
# STEP 8: VERIFY ACTUAL STRUCTURE (UPDATED)
# ============================================================================

def verify_oral_cancer_structure(base_path):
    """Verify the actual Oral Cancer dataset structure"""
    print("\n" + "=" * 70)
    print("STEP 8: VERIFYING DATASET STRUCTURE")
    print("=" * 70)

    print("\n🔍 Analyzing extracted dataset structure...\n")

    # Find the main dataset directory
    dataset_dirs = list(base_path.glob("Histopathological imaging database*"))

    if not dataset_dirs:
        print("❌ Dataset directory not found!")
        return None

    main_dir = dataset_dirs[0]
    print(f"✅ Found dataset directory: {main_dir.name}\n")

    # Check for First Set and Second Set
    first_set = main_dir / "First Set"
    second_set = main_dir / "Second Set"

    structure_info = {}

    if first_set.exists():
        print("📁 First Set (100x magnification):")

        # Find Normal and OSCC directories
        for subdir in first_set.iterdir():
            if subdir.is_dir():
                image_count = len(list(subdir.glob('*.jpg'))) + len(list(subdir.glob('*.png')))
                print(f"   ✅ {subdir.name}: {image_count} images")
                structure_info[subdir.name] = {
                    'path': subdir,
                    'count': image_count,
                    'magnification': '100x'
                }

    if second_set.exists():
        print("\n📁 Second Set (400x magnification):")

        # Find Normal and OSCC directories
        for subdir in second_set.iterdir():
            if subdir.is_dir():
                image_count = len(list(subdir.glob('*.jpg'))) + len(list(subdir.glob('*.png')))
                print(f"   ✅ {subdir.name}: {image_count} images")
                structure_info[subdir.name] = {
                    'path': subdir,
                    'count': image_count,
                    'magnification': '400x'
                }

    # Summary
    total_images = sum(info['count'] for info in structure_info.values())
    print(f"\n{'='*70}")
    print(f"📊 Total images found: {total_images}")
    print(f"✅ Dataset structure verified successfully!")

    print("\n💡 Dataset Organization:")
    print(f"   Base Path: {main_dir}")
    print(f"   - First Set/  (100x magnification)")
    print(f"   - Second Set/ (400x magnification)")

    print("\n" + "-" * 70)

    return structure_info

# ============================================================================
# STEP 9: LOAD AND DISPLAY SAMPLE IMAGES (NEW)
# ============================================================================

def display_sample_images(base_path):
    """Load and display sample images from the dataset"""
    print("\n" + "=" * 70)
    print("STEP 9: LOADING AND DISPLAYING SAMPLE IMAGES")
    print("=" * 70)

    # Find the main dataset directory
    dataset_dirs = list(base_path.glob("Histopathological imaging database*"))

    if not dataset_dirs:
        print("❌ Dataset directory not found!")
        return

    main_dir = dataset_dirs[0]

    # Define paths
    first_set = main_dir / "First Set"
    second_set = main_dir / "Second Set"

    # Find directories
    normal_100x_dirs = list(first_set.glob("*Normal*"))
    oscc_100x_dirs = list(first_set.glob("*OSCC*"))
    normal_400x_dirs = list(second_set.glob("*Normal*"))
    oscc_400x_dirs = list(second_set.glob("*OSCC*"))

    print("\n📸 Loading sample images...\n")

    # Collect sample images
    samples = []

    if normal_100x_dirs:
        images = list(normal_100x_dirs[0].glob('*.jpg'))[:2]
        for img in images:
            samples.append(('Normal 100x', img))

    if oscc_100x_dirs:
        images = list(oscc_100x_dirs[0].glob('*.jpg'))[:2]
        for img in images:
            samples.append(('OSCC 100x', img))

    if normal_400x_dirs:
        images = list(normal_400x_dirs[0].glob('*.jpg'))[:2]
        for img in images:
            samples.append(('Normal 400x', img))

    if oscc_400x_dirs:
        images = list(oscc_400x_dirs[0].glob('*.jpg'))[:2]
        for img in images:
            samples.append(('OSCC 400x', img))

    if not samples:
        print("❌ No sample images found!")
        return

    # Display images in a grid
    fig, axes = plt.subplots(4, 2, figsize=(15, 20))
    fig.suptitle('Oral Cancer Dataset - Sample Images', fontsize=16, fontweight='bold')

    for idx, (label, img_path) in enumerate(samples):
        row = idx // 2
        col = idx % 2

        try:
            # Load image
            img = mpimg.imread(str(img_path))

            # Display
            axes[row, col].imshow(img)
            axes[row, col].set_title(f'{label}\n{img_path.name}', fontsize=10)
            axes[row, col].axis('off')

            # Print image info
            print(f"✅ {label}: {img_path.name}")
            print(f"   Shape: {img.shape}, Size: {img_path.stat().st_size / (1024**2):.2f} MB")

        except Exception as e:
            axes[row, col].text(0.5, 0.5, f'Error loading\n{label}',
                              ha='center', va='center')
            axes[row, col].axis('off')
            print(f"❌ Error loading {label}: {str(e)}")

    plt.tight_layout()
    plt.show()

    print("\n" + "-" * 70)

    # Display detailed image information
    print("\n" + "=" * 70)
    print("IMAGE DETAILS")
    print("=" * 70)

    for label, img_path in samples:
        try:
            img = mpimg.imread(str(img_path))
            print(f"\n📸 {label}: {img_path.name}")
            print(f"   Path: {img_path.parent.name}")
            print(f"   Dimensions: {img.shape[1]}x{img.shape[0]} pixels")
            print(f"   Channels: {img.shape[2] if len(img.shape) > 2 else 1}")
            print(f"   Data type: {img.dtype}")
            print(f"   File size: {img_path.stat().st_size / (1024**2):.2f} MB")
            print(f"   Pixel value range: [{img.min():.3f}, {img.max():.3f}]")
        except Exception as e:
            print(f"   ❌ Error: {str(e)}")

    print("\n" + "-" * 70)

# ============================================================================
# STEP 10: CREATE DATASET PATHS REFERENCE
# ============================================================================

def create_dataset_paths_reference(base_path):
    """Create easy-to-use path references for the dataset"""
    print("\n" + "=" * 70)
    print("STEP 10: DATASET PATH REFERENCES")
    print("=" * 70)

    # Find the main dataset directory
    dataset_dirs = list(base_path.glob("Histopathological imaging database*"))

    if not dataset_dirs:
        print("❌ Dataset directory not found!")
        return None

    main_dir = dataset_dirs[0]

    # Create path dictionary
    paths = {}

    # First Set (100x)
    first_set = main_dir / "First Set"
    if first_set.exists():
        normal_100x = list(first_set.glob("*Normal*"))
        oscc_100x = list(first_set.glob("*OSCC*"))

        if normal_100x:
            paths['normal_100x'] = normal_100x[0]
        if oscc_100x:
            paths['oscc_100x'] = oscc_100x[0]

    # Second Set (400x)
    second_set = main_dir / "Second Set"
    if second_set.exists():
        normal_400x = list(second_set.glob("*Normal*"))
        oscc_400x = list(second_set.glob("*OSCC*"))

        if normal_400x:
            paths['normal_400x'] = normal_400x[0]
        if oscc_400x:
            paths['oscc_400x'] = oscc_400x[0]

    print("\n📋 Quick Access Paths:\n")
    print("```python")
    print("from pathlib import Path")
    print()
    print(f"# Base path")
    print(f"base_path = Path('{main_dir}')")
    print()

    for key, path in paths.items():
        print(f"# {key.replace('_', ' ').title()}")
        print(f"{key}_path = base_path / '{path.relative_to(main_dir)}'")
        image_count = len(list(path.glob('*.jpg')))
        print(f"# Contains {image_count} images")
        print()

    print("```")

    print("\n💡 Usage Example:")
    print("```python")
    print("# Load all normal 100x images")
    print("normal_100x_images = list(normal_100x_path.glob('*.jpg'))")
    print("print(f'Found {len(normal_100x_images)} normal 100x images')")
    print()
    print("# Load a specific image")
    print("import matplotlib.pyplot as plt")
    print("import matplotlib.image as mpimg")
    print()
    print("img = mpimg.imread(str(normal_100x_images[0]))")
    print("plt.imshow(img)")
    print("plt.title('Normal Oral Cavity - 100x')")
    print("plt.axis('off')")
    print("plt.show()")
    print("```")

    print("\n" + "-" * 70)

    return paths

# ============================================================================
# MAIN EXECUTION FUNCTION
# ============================================================================

def setup_oral_cancer_dataset_in_drive():
    """Main function to download and setup the Oral Cancer dataset"""

    print("=" * 70)
    print("🦷 ORAL CANCER DATASET SETUP IN GOOGLE DRIVE")
    print("=" * 70)
    print("\nThis script will:")
    print("  1. Mount Google Drive")
    print("  2. Download dataset from AWS S3 (~3GB)")
    print("  3. Save to: /Langraph Projects/Oral Cancer/")
    print("  4. Extract the dataset")
    print("  5. Delete ZIP file to save space")
    print("  6. Analyze and display file structure")
    print("  7. Verify dataset organization")
    print("  8. Display sample images")
    print("  9. Create path references")
    print("\n" + "=" * 70)

    # Dataset URL
    dataset_url = "https://prod-dcd-datasets-cache-zipfiles.s3.eu-west-1.amazonaws.com/ftmp4cvtmb-2.zip"

    try:
        # Step 1: Mount Google Drive
        mount_google_drive()

        # Step 2: Create directories
        base_path = create_directories()

        # Step 3: Download dataset
        zip_filename = base_path / "ftmp4cvtmb-2.zip"

        if not zip_filename.exists():
            download_success = download_dataset(dataset_url, zip_filename)

            if not download_success:
                raise Exception("Download failed!")
        else:
            print(f"\n✅ ZIP file already exists: {zip_filename}")
            print(f"📊 File size: {os.path.getsize(zip_filename) / (1024**3):.2f} GB")

        # Step 4: Extract dataset
        extract_success = extract_dataset(zip_filename, base_path)

        if not extract_success:
            raise Exception("Extraction failed!")

        # Step 5: Delete ZIP file
        delete_zip_file(zip_filename)

        # Step 6: Analyze directory structure
        analyze_directory_structure(base_path, max_depth=2)

        # Step 7: Show statistics
        show_dataset_statistics(base_path)

        # Step 8: Verify structure (UPDATED)
        structure_info = verify_oral_cancer_structure(base_path)

        # Step 9: Display sample images (NEW)
        display_sample_images(base_path)

        # Step 10: Create path references (NEW)
        dataset_paths = create_dataset_paths_reference(base_path)

        # Final summary
        print("\n" + "=" * 70)
        print("🎉 SETUP COMPLETE!")
        print("=" * 70)
        print(f"\n✅ Dataset location: {base_path}")
        print(f"✅ Dataset extracted and organized")
        print(f"✅ ZIP file deleted to save space")
        print(f"✅ Sample images displayed")

        print("\n📝 Dataset is ready for machine learning!")
        print("\n" + "=" * 70)

        return base_path, dataset_paths

    except Exception as e:
        print(f"\n❌ Error occurred: {str(e)}")
        import traceback
        traceback.print_exc()
        raise

In [3]:
# ============================================================================
# RUN THE SETUP
# ============================================================================

if __name__ == "__main__":
    dataset_path, paths = setup_oral_cancer_dataset_in_drive()

Output hidden; open in https://colab.research.google.com to view.